In [ ]:
from ABL import LocalizationSession, preprocessing
import os
from pathlib import Path

def _find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'pyproject.toml').exists():
            return str(p)
    return str(Path.cwd())

wrkdir = _find_repo_root()

### CPU or GPU

The notebook runs on the CPU by default, which works on any machine. With an NVIDIA GPU (CUDA), set `USE_GPU = True` for the fast mode:

- `compile_fit=True` compiles the model fit with `torch.compile`, making `localize()` several times faster.

The first session of a Python process spends extra time compiling during setup (up to ~20 s). If an option cannot be used on your machine, ABL prints a warning and falls back to the standard implementation.

In [ ]:
# Locate the example cortex model
example_model = os.path.join(wrkdir, "examples", "saved", "example_subject", "example_model.pkl")
_, _, cortex = preprocessing.load_example_model(example_model)

# Locate example trial data
example_trials_path = os.path.join(wrkdir, "examples", "saved", "example_subject", "example_efield_trials_and_responses.pkl")
efield_trials, responses = preprocessing.load_example_trials(example_trials_path)

# Where to save this session's outputs
save_path = os.path.join(wrkdir, "examples", "saved", "example_subject")

# Start localization instance
USE_GPU = False  # True: fast mode on an NVIDIA GPU (see above)

if USE_GPU:
    gpu_options = dict(device='cuda', compile_fit=True)
else:
    gpu_options = dict(device='cpu', compile_fit=False)

S = LocalizationSession(cortex=cortex, save_path=save_path, **gpu_options)

Run localization analysis. Takes a few minutes with `USE_GPU = True`, about 10x longer on the CPU.

In [ ]:
S.import_trial_data(efield_trials,responses)
S.localize(full=True)

In [ ]:
# End session to stop plotting app and free resources
S.end_session()